In [177]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
import tqdm
import pickle
from time import time_ns

from src.data_import import *
from src.train import train_one_epoch_TC, train_one_epoch
from src.model import EBM

In [178]:
class ToyEnergyHead(nn.Module):
    def __init__(self, mid=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, mid),
            nn.GELU(),
            nn.Linear(mid, mid),
            nn.GELU(),
            nn.Linear(mid, 1),
        )

    def forward(self, x):
        return self.net(x)



class ToyEBM(nn.Module):

    def __init__(self, mid_dim: int = 4, n_heads: int = 2) -> None:
        super(ToyEBM, self).__init__()
        self.mid_dim = mid_dim
        self.n_heads = n_heads
        self.device = None

        self.heads = nn.ModuleList(
            [ToyEnergyHead(mid=mid_dim) for _ in range(n_heads)]
        )

    def forward(self, x, head_idx=None):

        if head_idx is not None:
            h_o = self.heads[head_idx](x).squeeze(-1)

        else:
            h_o = torch.stack([head(x).squeeze(-1) for head in self.heads], dim=1)

        if h_o.dim() == 1:
            energy = h_o
        else:
            energy = h_o.sum(dim=1)

        return energy, h_o

In [179]:

def plot_model_density(
        model, 
        samples, 
        bins=100, 
        x_min=-2, 
        x_max=2, 
        device=torch.device("cpu")):
    
    model.eval()
    x = torch.as_tensor(samples, dtype=torch.float32, device=device).view(-1, 1)
    with torch.no_grad():
        y = model(x).detach().cpu().view(-1).numpy()

    plt.hist(y, bins=bins, density=True, range=(0,1))
    plt.title("Model density from samples")
    plt.show()

In [ ]:
import torch
from torch.utils.data import TensorDataset

def sample_from_1d_density(density_fn, n_samples, x_max=20.0, batch_size=32, M=2, device="cpu"):
    samples = []
    while len(samples) < n_samples:
        x = (2 * x_max) * torch.randn(batch_size, 5, device=device)
        u = M * torch.rand(batch_size, 1, device=device)
        with torch.no_grad():
            fx = density_fn(x)
        accepted = x[u <= fx]
        if accepted.numel() > 0:
            samples.append(accepted.view(-1))

    samples = torch.cat(samples)[:n_samples]
    return TensorDataset(samples.unsqueeze(1).unsqueeze(0), torch.empty_like(samples.unsqueeze(1).unsqueeze(0)))


In [181]:
def strange_density(x: torch.Tensor) -> torch.Tensor:
    t1 = torch.exp(-x**2/2)*(torch.cos(3*x)**2)
    t2 = F.relu(x) - F.relu(x-1) - F.relu(torch.sign(x-1))
    return (t1 + t2)/1.7533141564

class StrangeModule(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, x):
        return strange_density(x)

In [182]:
s = StrangeModule()
samples = sample_from_1d_density(s, 10000)
plot_model_density(s, samples)

ValueError: only one element tensors can be converted to Python scalars

In [ ]:
from src.train import train_one_epoch_TC
from src.information import TotalCorrelationEstimator
from src.sampler import ReplaySampler

model = ToyEBM()
optim = torch.optim.Adam(model.parameters(), lr=1e-3)
sampler = ReplaySampler(model, img_shape=(1,1), buffer_size=3, noise_fraction=0.005)
tc_estim = TotalCorrelationEstimator(1, hidden_dim=3, lr=1e-3)

In [185]:
next(iter(samples))[0].shape

torch.Size([10000, 1])

In [ ]:
for epoch in range(16):
    epoch_loss, _ = train_one_epoch_TC(
        model, 
        sampler, 
        samples, 
        optim,

        sample_steps=50,
        sample_step_size=10.0,
        sample_noise_std=0.005,
        energy_reg=1e-1,
        tc_regularizations=1e-2,
        
        tc_estimator=tc_estim,
        device=torch.device("cpu")
    )
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")

    if epoch % 5 == 0:
        pass # torch.save(model.state_dict(), f"./models/lfw/EBM_{epoch}_epochs_{len(model.heads)}_heads_{IMAGE_SHAPE}_shape_{time_ns()}t.pth")


  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]


IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)